# Step 5: Decision Engine
## Menghasilkan Keputusan Otomatis Platform

**Tujuan**:
- Baca hasil Scoring Engine dari `product_scores.csv`
- Terapkan threshold rules untuk 3 kategori keputusan
- Hasilkan kolom decision & action untuk setiap produk
- Output: `product_decisions.csv` (file output final sistem)

---

## Threshold Rules

| Kondisi | Keputusan | Aksi Platform |
|---------|-----------|---------------|
| score ≥ 0.65 | PERTAHANKAN | Listing tetap aktif, tidak ada tindakan |
| 0.35 ≤ score < 0.65 | EVALUASI | Notifikasi seller: perbaiki aspek [weakness_flag] |
| score < 0.35 | TARIK | Listing di-suspend, review ulang 30 hari |

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up paths
BASE_PATH = Path('../')                                                    # ← wajib ada
PRODUCT_SCORES = BASE_PATH / 'outputs' / 'product_scores.csv'             # ← 60K data
OUTPUT_PATH = BASE_PATH / 'outputs' / 'product_decisions.csv'

print(f"Checking files...")
print(f"Product Scores exists: {PRODUCT_SCORES.exists()}")


Checking files...
Product Scores exists: True


## 1. Load Product Scores

In [3]:
# Load product scores dari Step 4
df_decisions = pd.read_csv(PRODUCT_SCORES)

print(f"Shape: {df_decisions.shape}")
print(f"\nColumns: {df_decisions.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df_decisions.head())

print(f"\nProduct Score Statistics:")
print(df_decisions['product_score'].describe())

Shape: (23426, 8)

Columns: ['product_id', 'review_count', 'aspect_harga_score', 'aspect_kualitas_score', 'aspect_pengiriman_score', 'aspect_layanan_score', 'product_score', 'weakness_flags']

First 5 rows:
   product_id  review_count  aspect_harga_score  aspect_kualitas_score  \
0  B000UC2F7O             1                 1.0                    1.0   
1  9983891204             1                 1.0                    1.0   
2  B000HHM5EA             1                 1.0                    1.0   
3  B009KUE71O             1                 1.0                    1.0   
4  B0001F9KL4             1                 1.0                    1.0   

   aspect_pengiriman_score  aspect_layanan_score  product_score weakness_flags  
0                      1.0                   1.0            1.0           NONE  
1                      1.0                   1.0            1.0           NONE  
2                      1.0                   1.0            1.0           NONE  
3                      1

## 2. Apply Decision Threshold

In [4]:
# Define thresholds
THRESHOLD_KEEP = 0.65       # score >= 0.65 = PERTAHANKAN
THRESHOLD_EVALUATE = 0.35   # 0.35 <= score < 0.65 = EVALUASI
# score < 0.35 = TARIK

def assign_decision(score):
    """
    Assign decision based on product_score threshold.
    """
    if score >= THRESHOLD_KEEP:
        return 'PERTAHANKAN'
    elif score >= THRESHOLD_EVALUATE:
        return 'EVALUASI'
    else:
        return 'TARIK'

# Apply decision logic
df_decisions['decision'] = df_decisions['product_score'].apply(assign_decision)

print("Decision Distribution:")
print(df_decisions['decision'].value_counts())
print(f"\nPercentage:")
print((df_decisions['decision'].value_counts(normalize=True) * 100).round(2))

Decision Distribution:
decision
EVALUASI       16817
TARIK           3671
PERTAHANKAN     2938
Name: count, dtype: int64

Percentage:
decision
EVALUASI       71.79
TARIK          15.67
PERTAHANKAN    12.54
Name: proportion, dtype: float64


## 3. Generate Action Items

In [5]:
def generate_action(row):
    """
    Generate platform action based on decision.
    """
    decision = row['decision']
    weakness = row['weakness_flags']
    
    if decision == 'PERTAHANKAN':
        return 'Listing tetap aktif. Tidak ada tindakan.'
    elif decision == 'EVALUASI':
        return f'NOTIFIKASI SELLER: Perbaiki aspek [{weakness}]. Target ulang 14 hari.'
    else:  # TARIK
        return 'Listing di-suspend. Review ulang 30 hari.'

# Apply action generation
df_decisions['action'] = df_decisions.apply(generate_action, axis=1)

print("Sample Actions:")
print(df_decisions[['product_id', 'product_score', 'decision', 'weakness_flags', 'action']].head(15))

Sample Actions:
    product_id  product_score     decision weakness_flags  \
0   B000UC2F7O            1.0  PERTAHANKAN           NONE   
1   9983891204            1.0  PERTAHANKAN           NONE   
2   B000HHM5EA            1.0  PERTAHANKAN           NONE   
3   B009KUE71O            1.0  PERTAHANKAN           NONE   
4   B0001F9KL4            1.0  PERTAHANKAN           NONE   
5   B00DGH9ROO            1.0  PERTAHANKAN           NONE   
6   B00GKONE2G            1.0  PERTAHANKAN           NONE   
7   B000EO9MD8            1.0  PERTAHANKAN           NONE   
8   B00S1ZWIZG            1.0  PERTAHANKAN           NONE   
9   B005NB8CN8            1.0  PERTAHANKAN           NONE   
10  B00AQU2VLA            1.0  PERTAHANKAN           NONE   
11  B0007U0IGE            1.0  PERTAHANKAN           NONE   
12  B001NNUBNY            1.0  PERTAHANKAN           NONE   
13  B00R43OK66            1.0  PERTAHANKAN           NONE   
14  B001CRPIXE            1.0  PERTAHANKAN           NONE   

       

## 4. Assign Priority Level

In [6]:
def assign_priority(row):
    """
    Assign priority based on decision and weakness count.
    """
    decision = row['decision']
    weakness = row['weakness_flags']
    score = row['product_score']
    
    if decision == 'TARIK':
        if score < 0.25:
            return 'CRITICAL'
        else:
            return 'HIGH'
    elif decision == 'EVALUASI':
        weakness_count = len(weakness.split(',')) if weakness != 'NONE' else 0
        return 'MEDIUM' if weakness_count < 2 else 'HIGH'
    else:  # PERTAHANKAN
        return 'LOW'

df_decisions['priority'] = df_decisions.apply(assign_priority, axis=1)

print("Priority Distribution:")
print(df_decisions['priority'].value_counts())
print(f"\nCritical Items:")
print(df_decisions[df_decisions['priority'] == 'CRITICAL'].shape[0])

Priority Distribution:
priority
MEDIUM      15439
HIGH         4027
LOW          2938
CRITICAL     1022
Name: count, dtype: int64

Critical Items:
1022


## 5. Add Timeline

In [7]:
today = datetime.now().date()

def assign_deadline(row):
    decision = row['decision']
    if decision == 'PERTAHANKAN':
        return None  # No deadline
    elif decision == 'EVALUASI':
        deadline = today + timedelta(days=14)
        return deadline
    else:  # TARIK
        deadline = today + timedelta(days=30)
        return deadline

df_decisions['deadline'] = df_decisions.apply(assign_deadline, axis=1)
df_decisions['created_date'] = today

print(f"Deadlines assigned:")
print(df_decisions[df_decisions['deadline'].notna()][['product_id', 'decision', 'deadline']].head(10))

Deadlines assigned:
      product_id  decision    deadline
2938  B000QIWYSM  EVALUASI  2026-06-22
2939  B000R9J5OG  EVALUASI  2026-06-22
2940  B000X1XP3U  EVALUASI  2026-06-22
2941  B000TB0RY4  EVALUASI  2026-06-22
2942  B000CS1JRS  EVALUASI  2026-06-22
2943  B001A5PDKQ  EVALUASI  2026-06-22
2944  B00A39PPI0  EVALUASI  2026-06-22
2945  B0013OWPVO  EVALUASI  2026-06-22
2946  B002PY7H3M  EVALUASI  2026-06-22
2947  B00155V210  EVALUASI  2026-06-22


## 6. Final Output

In [8]:
# Select final output columns
output_cols = [
    'product_id',
    'review_count',
    'aspect_harga_score',
    'aspect_kualitas_score',
    'aspect_pengiriman_score',
    'aspect_layanan_score',
    'product_score',
    'weakness_flags',
    'decision',
    'priority',
    'action',
    'deadline',
    'created_date'
]

df_final = df_decisions[output_cols].copy()

# Sort by priority and score
priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
df_final['priority_num'] = df_final['priority'].map(priority_order)
df_final = df_final.sort_values(['priority_num', 'product_score'], ascending=[True, False]).drop('priority_num', axis=1)

print(f"\nFinal Output Shape: {df_final.shape}")
print(f"\n CRITICAL & HIGH Priority Items:")
print(df_final[df_final['priority'].isin(['CRITICAL', 'HIGH'])][['product_id', 'product_score', 'decision', 'priority', 'weakness_flags']].head(20))

print(f"\n Sample PERTAHANKAN Products:")
print(df_final[df_final['decision'] == 'PERTAHANKAN'][['product_id', 'product_score', 'decision', 'weakness_flags']].head(5))


Final Output Shape: (23426, 13)

 CRITICAL & HIGH Priority Items:
       product_id  product_score decision  priority  \
22404  B003Z5W7CG       0.250000    TARIK  CRITICAL   
22405  B00ELAM8M8       0.250000    TARIK  CRITICAL   
22406  B00ECRPOKS       0.250000    TARIK  CRITICAL   
22407  B002CSUZSU       0.250000    TARIK  CRITICAL   
22408  B002BC32OG       0.250000    TARIK  CRITICAL   
22409  B0015AFWC0       0.250000    TARIK  CRITICAL   
22410  B00005NHHB       0.250000    TARIK  CRITICAL   
22411  B001C3TWIA       0.250000    TARIK  CRITICAL   
22412  B00GXERG2C       0.250000    TARIK  CRITICAL   
22413  B00B9DQ2QI       0.250000    TARIK  CRITICAL   
22414  B000067O9B       0.250000    TARIK  CRITICAL   
22415  B005ODHJFM       0.250000    TARIK  CRITICAL   
22416  B001TP4RRC       0.250000    TARIK  CRITICAL   
22417  B003JHJL38       0.250000    TARIK  CRITICAL   
22418  B005OB1TU0       0.229167    TARIK  CRITICAL   
22419  B001CJ9392       0.225000    TARIK  CRITICAL  

In [9]:
# Save to CSV
df_final.to_csv(OUTPUT_PATH, index=False)
print(f" Saved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024:.2f} KB")
print(f"\n Summary:")
print(f"  - Total Products: {len(df_final)}")
print(f"  - PERTAHANKAN: {(df_final['decision'] == 'PERTAHANKAN').sum()} ({(df_final['decision'] == 'PERTAHANKAN').sum() / len(df_final) * 100:.1f}%)")
print(f"  - EVALUASI: {(df_final['decision'] == 'EVALUASI').sum()} ({(df_final['decision'] == 'EVALUASI').sum() / len(df_final) * 100:.1f}%)")
print(f"  - TARIK: {(df_final['decision'] == 'TARIK').sum()} ({(df_final['decision'] == 'TARIK').sum() / len(df_final) * 100:.1f}%)")

 Saved to: ..\outputs\product_decisions.csv
File size: 3362.09 KB

 Summary:
  - Total Products: 23426
  - PERTAHANKAN: 2938 (12.5%)
  - EVALUASI: 16817 (71.8%)
  - TARIK: 3671 (15.7%)


## 7. Decision Analysis

In [10]:
print("DECISION ENGINE - ANALYSIS REPORT")

print(f"\n Decision Summary:")
print(df_final['decision'].value_counts())

print(f"\n Priority Distribution:")
print(df_final['priority'].value_counts())

print(f"\n Decision vs Priority Crosstab:")
print(pd.crosstab(df_final['decision'], df_final['priority'], margins=True))

print(f"\n Most Common Weaknesses:")
weakness_list = []
for weaknesses in df_final[df_final['weakness_flags'] != 'NONE']['weakness_flags']:
    weakness_list.extend(weaknesses.split(','))
weakness_df = pd.Series(weakness_list).value_counts()
print(weakness_df)

print(f"\n Deadline Summary:")
print(f"  - Items dengan deadline: {df_final['deadline'].notna().sum()}")
print(f"  - Evaluation deadline (14 hari): {(df_final['decision'] == 'EVALUASI').sum()}")
print(f"  - Suspension review (30 hari): {(df_final['decision'] == 'TARIK').sum()}")

DECISION ENGINE - ANALYSIS REPORT

 Decision Summary:
decision
EVALUASI       16817
TARIK           3671
PERTAHANKAN     2938
Name: count, dtype: int64

 Priority Distribution:
priority
MEDIUM      15439
HIGH         4027
LOW          2938
CRITICAL     1022
Name: count, dtype: int64

 Decision vs Priority Crosstab:
priority     CRITICAL  HIGH   LOW  MEDIUM    All
decision                                        
EVALUASI            0  1378     0   15439  16817
PERTAHANKAN         0     0  2938       0   2938
TARIK            1022  2649     0       0   3671
All              1022  4027  2938   15439  23426

 Most Common Weaknesses:
HARGA         5251
KUALITAS      4345
LAYANAN       4112
PENGIRIMAN    3427
Name: count, dtype: int64

 Deadline Summary:
  - Items dengan deadline: 20488
  - Evaluation deadline (14 hari): 16817
  - Suspension review (30 hari): 3671


In [11]:
print(" STEP 5 COMPLETE - Decision Engine")
print(f"\nOutput file ready: product_decisions.csv")
print(f"Next step: Step 6 - Visualisasi & Report")

 STEP 5 COMPLETE - Decision Engine

Output file ready: product_decisions.csv
Next step: Step 6 - Visualisasi & Report
